# Customer Segmentation - Exploratory Analysis

This notebook is the exploratory companion to the production pipeline in `src/retail_segmentation/`.
It walks through the same steps interactively: load transactions, engineer RFM features, pick `k`, fit K-means, and inspect the resulting segments.

Run `retail-segment train` once first (see the project README) so `data/raw/transactions.csv` exists.

In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import pandas as pd

from retail_segmentation.config import load_config
from retail_segmentation.data import load_transactions
from retail_segmentation.features import compute_rfm
from retail_segmentation.evaluate import score_k_range, best_k
from retail_segmentation.model import FEATURE_COLUMNS, train
from retail_segmentation.segments import label_segments, segment_summary

cfg = load_config("../configs/config.yaml")

## 1. Load transactions and compute RFM features

In [ ]:
transactions = load_transactions(cfg["data"]["raw_path"])
rfm = compute_rfm(transactions, snapshot_date=cfg["data"]["snapshot_date"])
rfm.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, FEATURE_COLUMNS, strict=True):
    ax.hist(rfm[col], bins=40)
    ax.set_title(col)
fig.suptitle("RFM distributions are right-skewed - this is why the pipeline log-transforms them")
fig.tight_layout()

## 2. Choose k with the elbow method + silhouette score

In [ ]:
k_min, k_max = cfg["model"]["k_range"]
scores = score_k_range(rfm, k_min, k_max, random_state=cfg["random_state"])
k = best_k(scores)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(scores["k"], scores["inertia"], marker="o")
ax1.set_title("Elbow method")
ax2.plot(scores["k"], scores["silhouette"], marker="o", color="darkorange")
ax2.set_title("Silhouette score")
fig.tight_layout()
print(f"Chosen k = {k}")

## 3. Fit K-means and inspect the segments

In [ ]:
pipeline = train(rfm, n_clusters=k, random_state=cfg["random_state"])
rfm["cluster"] = pipeline.named_steps["kmeans"].labels_
labeled = label_segments(rfm)
segment_summary(labeled)

In [ ]:
transformed = pipeline[:-1].transform(labeled[FEATURE_COLUMNS])
from sklearn.decomposition import PCA
coords = PCA(n_components=2, random_state=42).fit_transform(transformed)

plt.figure(figsize=(6, 5))
plt.scatter(coords[:, 0], coords[:, 1], c=labeled["cluster"], cmap="tab10", s=12, alpha=0.7)
plt.title("Segments in PCA-reduced feature space")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()